# Design / Outfit Transfer — avaliacao A/B (run_003)

Linha **experimental**, 100% **nao-generativa**. Nao substitui FLUX, Flow 01,
quality gates nem o pipeline oficial. Nao produz `master.png`.

Ver `docs/research/2026-09-09-design-transfer.md`.

* **A** = controle: transformacao global + composicao
* **B** = mascaras separadas + TPS por regiao + composicao
* **C** = NAO implementada (sem editor mascarado open-weight, comercial,
  com referencia nativa e leve para T4)

Fonte do design: `full_body.png` + mascaras. **`outfit.png` nao e usado**
(e recorte retangular com pele e fundo).

Garantia verificada: **`outside_mask_pixel_difference == 0`** — fora da
mascara a Run 003 fica intacta.

> A escolha artistica e **humana**. Este notebook nao elege vencedor.


In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
#@markdown Instala as dependencias (todas permissivas) e localiza o repo.
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
ATUALIZAR_REPO = True  #@param {type:"boolean"}
#@markdown Mantenha marcado: garante que o runtime nao rode codigo antigo.

import subprocess, sys, os
from pathlib import Path

def sh(*a):
    print("$", " ".join(a))
    subprocess.run(a, check=False)

IN_COLAB = "google.colab" in sys.modules
# scikit-image >= 0.25 basta: o codigo detecta a API por versao
# (0.25.x e 0.26 divergem em remove_small_* e ThinPlateSplineTransform).
sh(sys.executable, "-m", "pip", "-q", "install",
   "numpy", "pillow", "scikit-image>=0.25")

ROOT = None
for c in (Path.cwd(), *Path.cwd().parents):
    if (c / "scripts" / "chibi").is_dir():
        ROOT = c
        break
if ROOT is None and Path("ChibiCreate/scripts/chibi").is_dir():
    ROOT = Path("ChibiCreate").resolve()
if ROOT is None:
    sh("git", "clone", "--branch", BRANCH, REPO_URL, "ChibiCreate")
    ROOT = Path("ChibiCreate").resolve()
os.chdir(ROOT)

# SEMPRE atualizar: um clone antigo do runtime deixaria o notebook rodando
# codigo obsoleto e reproduzindo bugs ja corrigidos.
if ATUALIZAR_REPO:
    sh("git", "fetch", "--quiet", "origin", BRANCH)
    sh("git", "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")

sys.path.insert(0, str(ROOT))

# limpar modulos ja importados, senao o Python reusa a versao velha da memoria
for _m in [m for m in sys.modules if m.startswith("scripts.chibi")]:
    del sys.modules[_m]

print("\nrepo:", ROOT)
subprocess.run(["git", "log", "--oneline", "-1"])

import numpy, PIL, skimage
print("numpy", numpy.__version__, "| pillow", PIL.__version__,
      "| scikit-image", skimage.__version__)
print("licencas: BSD-3-Clause / MIT-CMU / BSD-3-Clause  (todas comerciais)")


In [ ]:
#@title 2. Carregar Run 003 e full_body { display-mode: "form" }
#@markdown `run_003/output.png` **nao esta versionado** (`experiments/**/*.png`
#@markdown e gitignored). Faca upload dele. `full_body.png` vem do repo.
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
RUN_003_PATH = ""  #@param {type:"string"}
#@markdown Deixe vazio para abrir o seletor de upload.

import sys
from pathlib import Path
import numpy as np
from PIL import Image

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.chibi import design_transfer as dt

REF = ROOT / "characters" / CHARACTER_ID / "reference"
FULL_BODY = REF / "full_body.png"
assert FULL_BODY.exists(), f"nao encontrei {FULL_BODY}"

WORK = ROOT / "experiments" / "design_transfer" / "run_003"
(WORK / "masks").mkdir(parents=True, exist_ok=True)

base_path = Path(RUN_003_PATH) if RUN_003_PATH else None
if base_path is None or not base_path.exists():
    try:
        from google.colab import files
        print("Selecione run_003/output.png:")
        up = files.upload()
        name = next(iter(up))
        base_path = WORK / "run_003_input.png"
        base_path.write_bytes(up[name])
    except ImportError:
        raise SystemExit("Fora do Colab: preencha RUN_003_PATH.")

BASE = dt.load_rgba(base_path)
SOURCE = dt.load_rgba(FULL_BODY)
print("Run 003   :", base_path, BASE.size, BASE.mode)
print("full_body :", FULL_BODY, SOURCE.size, SOURCE.mode)

if BASE.size != SOURCE.size:
    print(f"\n[nota] resolucoes diferentes; o warp mapeia a caixa do sujeito,")
    print("       entao isso e esperado e nao impede a composicao.")


In [ ]:
#@title 3. Ver as entradas (1 = Run 003, 2 = full_body) { display-mode: "form" }
import matplotlib.pyplot as plt

def flat(img, bg=(255, 255, 255)):
    c = Image.new("RGB", img.size, bg)
    c.paste(img, (0, 0), img)
    return c

fig, ax = plt.subplots(1, 2, figsize=(11, 6))
ax[0].imshow(flat(BASE));   ax[0].set_title("1. Run 003 (base preservada)")
ax[1].imshow(flat(SOURCE)); ax[1].set_title("2. full_body (fonte do design)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 4. Construir as mascaras (3, 4, 5) { display-mode: "form" }
#@markdown Cada mascara = **cor AND regiao geometrica AND alpha**.
#@markdown Nunca cor isolada: capa e cabelo tem distancia RGB ~29.5.
MIN_AREA = 64  #@param {type:"integer"}

MASKS = dt.build_masks(SOURCE, min_area=MIN_AREA)
subj = dt.subject_mask(SOURCE)
total = int(np.count_nonzero(subj))

print(f"{'regiao':14}{'pixels':>9}{'% sujeito':>11}  rigidez")
for name in ("roupa", "capa", "ornamentos"):
    n = int(np.count_nonzero(MASKS[name]))
    print(f"{name:14}{n:9d}{100*n/total:10.1f}%  {dt.REGION_RIGIDITY[name]}")

fig, ax = plt.subplots(1, 3, figsize=(15, 6))
for a, name, num in zip(ax, ("roupa", "capa", "ornamentos"), (3, 4, 5)):
    a.imshow(MASKS[name], cmap="gray")
    a.set_title(f"{num}. mascara: {name}")
    a.axis("off")
plt.tight_layout(); plt.show()

MASK_PATHS = {}
for name, m in MASKS.items():
    p = WORK / "masks" / f"source_{name}.png"
    Image.fromarray((m * 255).astype(np.uint8), "L").save(p)
    MASK_PATHS[name] = p
    print("salvo:", p.relative_to(ROOT), "| sha256", dt.mask_sha256(m)[:16])


In [ ]:
#@title 6. Overlay das tres mascaras sobre a Run 003 { display-mode: "form" }
#@markdown azul = roupa · magenta = capa · dourado = ornamentos
OVERLAY_ALPHA = 0.45  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

ov_src = dt.overlay_masks(SOURCE, MASKS, alpha=OVERLAY_ALPHA)

# as mascaras nascem no espaco do full_body; para ve-las sobre a Run 003
# aplicamos a mesma transformacao global que a composicao usa.
sb = dt.bbox_of(dt.subject_mask(SOURCE))
db = dt.bbox_of(dt.subject_mask(BASE))
shape = np.array(BASE).shape[:2]
warped = {
    k: dt.warp_affine(m.astype(float), sb, db, shape, order=0) > 0.5
    for k, m in MASKS.items()
}
ov_base = dt.overlay_masks(BASE, warped, alpha=OVERLAY_ALPHA)

fig, ax = plt.subplots(1, 2, figsize=(11, 6))
ax[0].imshow(flat(ov_src));  ax[0].set_title("mascaras sobre full_body")
ax[1].imshow(flat(ov_base)); ax[1].set_title("6. overlay sobre a Run 003")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

p = WORK / "masks" / "overlay_on_run003.png"
ov_base.save(p); print("salvo:", p.relative_to(ROOT))
print("\n[HUMAN REVIEW REQUIRED] confira o overlay antes de compor.")
print("Vazamento de mascara aqui vira artefato no resultado.")


In [ ]:
#@title 7. Executar A e B { display-mode: "form" }
TPS_GRID = 5      #@param {type:"slider", min:3, max:9, step:1}
FEATHER_A = 0.0   #@param {type:"slider", min:0.0, max:2.0, step:0.1}
FEATHER_B = 0.6   #@param {type:"slider", min:0.0, max:2.0, step:0.1}

RESULTS = {}
RESULTS["A"] = dt.run_variant_a(BASE, SOURCE, MASKS, feather=FEATHER_A)
RESULTS["B"] = dt.run_variant_b(BASE, SOURCE, MASKS, grid=TPS_GRID,
                                feather=FEATHER_B)

OUT_PATHS = {}
for k, r in RESULTS.items():
    p = WORK / f"variant_{k.lower()}.png"
    r.image.save(p)
    OUT_PATHS[k] = p
    d = r.metrics["outside_mask_pixel_difference"]
    ok = "OK" if d == 0 else "FALHOU"
    print(f"[{k}] {r.params['algorithm']}")
    print(f"     {r.elapsed_s}s | pixels alterados {r.metrics['pixels_changed']}")
    print(f"     outside_mask_pixel_difference = {d}  -> {ok}")
    assert d == 0, f"variante {k} alterou pixels fora da mascara"
print("\nGarantia confirmada: fora da mascara a Run 003 esta intacta.")


In [ ]:
#@title 8. Comparacao lado a lado { display-mode: "form" }
fig, ax = plt.subplots(1, 3, figsize=(16, 7))
ax[0].imshow(flat(BASE));                 ax[0].set_title("Run 003 (base)")
ax[1].imshow(flat(RESULTS["A"].image));   ax[1].set_title("A — global (controle)")
ax[2].imshow(flat(RESULTS["B"].image));   ax[2].set_title("B — mascaras + TPS")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

p = WORK / "comparison.png"
w, h = BASE.size
strip = Image.new("RGB", (w * 3, h), (255, 255, 255))
for i, im in enumerate([BASE, RESULTS["A"].image, RESULTS["B"].image]):
    strip.paste(flat(im.resize(BASE.size)), (i * w, 0))
strip.save(p); print("salvo:", p.relative_to(ROOT))


In [ ]:
#@title 9. Metricas, hashes e recipe { display-mode: "form" }
import json

INPUTS = {"run_003": base_path, "full_body": FULL_BODY}
RECIPES = {}
for k, r in RESULTS.items():
    rec = dt.build_recipe(r, INPUTS, OUT_PATHS[k], MASK_PATHS)
    RECIPES[k] = rec
    p = WORK / f"recipe_variant_{k.lower()}.json"
    p.write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    print(f"--- variante {k} ---")
    print("  artifact_sha256    ", rec["output"]["artifact_sha256"][:32])
    print("  output_pixel_sha256", rec["output"]["output_pixel_sha256"][:32])
    print("  outside_mask_diff  ", rec["metrics"]["outside_mask_pixel_difference"])
    print("  recipe:", p.relative_to(ROOT))

same = (RECIPES["A"]["output"]["output_pixel_sha256"]
        == RECIPES["B"]["output"]["output_pixel_sha256"])
print("\nA e B produziram o mesmo pixel hash?", same)
if same:
    print("  -> mascaras separadas nao acrescentaram nada; investigar.")


In [ ]:
#@title 10. Checklist de avaliacao humana + ZIP { display-mode: "form" }
#@markdown O agente **nao** escolhe vencedor. Preencha voce.
import shutil

print("""
EIXOS (por variante):
  STYLE              coerencia com o estilo chibi da Run 003
  IDENTITY           rosto, olhos, cabelo, chifres, expressao intactos
  DESIGN_PRESERVATION  <-- EIXO PRINCIPAL
                     capa, gola, ornamentos dourados e acessorios
                     reconheciveis como o design ORIGINAL

ARTEFATOS A PROCURAR:
  [ ] seams / bordas artificiais na transicao
  [ ] maos cobertas ou incorretas
  [ ] bracos ou pernas deformados
  [ ] textura inconsistente com a base
  [ ] shading incompativel (fonte tem sombreado realista; base e chibi)
  [ ] ornamentos dourados esticados ou quebrados
  [ ] vazamento de mascara (cabelo tratado como capa)

PERGUNTA DE ACEITE:
  E a personagem da Run 003 usando o design original,
  ou virou outra personagem?

Sem OVERALL automatico. A decisao e sua.
""")

zip_base = WORK.parent / "design_transfer_run_003"
shutil.make_archive(str(zip_base), "zip", WORK)
print("ZIP:", zip_base.with_suffix(".zip"))
try:
    from google.colab import files
    files.download(str(zip_base.with_suffix(".zip")))
except Exception:
    pass
